# 06 — Validación

**Fase 6 del proyecto ISO 20022 en Redes DLT.**

Contrasta las predicciones de la Fase 5 contra los valores reales de FRED y evalúa la hipótesis anticipatoria mediante:

1. **Métricas de error** sobre el conjunto de test (MSE, RMSE, MAE, R²).
2. **Test de causalidad de Granger** para verificar si las métricas on-chain *preceden* al indicador macro.
3. **Backtesting como clasificación de eventos** de estrés (matriz de confusión, precision, recall, F1).

Los reportes se guardan en `outputs/reportes/`.

# 1. Configuración

In [ ]:
DEBUG: bool = True       # activa prints de verificación
SAVE_DATA: bool = False  # activa guardado de reportes

# 2. Imports

In [ ]:
import os
import glob
import json

import pandas as pd
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score,
    confusion_matrix)

# statsmodels se importa dentro de test_granger para no
# exigirlo si esa sección no se ejecuta

# 3. Paths y Constantes

In [ ]:
# Paths de entrada / salida
BASE_DIR: str = "tesis"
ARCHIVO_PRED: str = os.path.join(
    BASE_DIR, "outputs", "reportes", "predicciones.csv")
ARCHIVO_ONCHAIN: str = os.path.join(
    BASE_DIR, "data", "processed", "txs_etiquetadas.csv")
DIR_EXTERNAL: str = os.path.join(
    BASE_DIR, "data", "external")
DIR_REPORTES: str = os.path.join(
    BASE_DIR, "outputs", "reportes")

# Constantes de validación
SERIE_OBJETIVO: str = "STLFSI4"  # índice de estrés (FRED)
MAX_LAG_GRANGER: int = 7        # rezagos del test de Granger
CUANTIL_EVENTO: float = 0.75    # umbral de evento de estrés
METRICAS_GRANGER: list[str] = [
    "n_txs", "monto_total", "fee_total"]
COLUMNAS: list[str] = [
    "tx_id", "red", "timestamp", "emisor", "receptor",
    "amount", "currency", "fee", "memo_raw",
    "memo_decoded", "iso_code", "iso_metodo",
]

# 4. Funciones

### 4.a — Carga de predicciones, on-chain y FRED

In [ ]:
def cargar_predicciones(ruta: str) -> pd.DataFrame:
    """Carga el CSV de predicciones de la Fase 5."""
    if not os.path.exists(ruta):
        if DEBUG:
            print(f"[DEBUG] no existe {ruta}")
        return pd.DataFrame()
    df = pd.read_csv(ruta)
    df["fecha"] = pd.to_datetime(
        df["fecha"], utc=True, errors="coerce")
    return df


def cargar_onchain(ruta: str) -> pd.DataFrame:
    """Carga el dataset etiquetado y parsea el timestamp."""
    if not os.path.exists(ruta):
        if DEBUG:
            print(f"[DEBUG] no existe {ruta}")
        return pd.DataFrame(columns=COLUMNAS)
    df = pd.read_csv(ruta)
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], utc=True, errors="coerce")
    return df


def cargar_fred_ultimo(directorio: str) -> dict:
    """Carga el JSON de FRED más reciente del directorio."""
    archivos = sorted(glob.glob(
        os.path.join(directorio, "fred_*.json")))
    if not archivos:
        return {}
    with open(archivos[-1], encoding="utf-8") as f:
        return json.load(f)

### 4.b — Construcción de series diarias

In [ ]:
def metricas_onchain(df: pd.DataFrame) -> pd.DataFrame:
    """Métricas on-chain diarias para el test de Granger."""
    base = df.dropna(subset=["timestamp"]).copy()
    if base.empty:
        return pd.DataFrame()
    base["fecha"] = base["timestamp"].dt.floor("D")
    return base.groupby("fecha").agg(
        n_txs=("tx_id", "count"),
        monto_total=("amount", "sum"),
        fee_total=("fee", "sum"),
    )


def construir_objetivo(fred: dict, serie: str) -> pd.Series:
    """Construye la serie objetivo (nivel) desde FRED."""
    obs = fred.get(serie)
    if not obs and fred:
        obs = next(iter(fred.values()))
    if not obs:
        return pd.Series(dtype=float, name="objetivo")
    tabla = pd.DataFrame(obs)
    tabla["date"] = pd.to_datetime(tabla["date"], utc=True)
    tabla["value"] = pd.to_numeric(
        tabla["value"], errors="coerce")
    serie_obj = (
        tabla.dropna(subset=["value"])
        .set_index("date")["value"]
    )
    return serie_obj.rename("objetivo")


def alinear_diario(objetivo: pd.Series,
                   idx: pd.Index) -> pd.Series:
    """Reindexa el objetivo al índice diario con ffill."""
    return (
        objetivo.reindex(objetivo.index.union(idx))
        .sort_index().ffill().reindex(idx)
    )

### 4.c — Métricas de error

In [ ]:
def metricas_error(df: pd.DataFrame) -> dict:
    """MSE, RMSE, MAE y R2 entre y_real e y_pred."""
    if df.empty:
        return {}
    y = df["y_real"]
    p = df["y_pred"]
    mse = mean_squared_error(y, p)
    return {
        "MSE": mse,
        "RMSE": mse ** 0.5,
        "MAE": mean_absolute_error(y, p),
        "R2": r2_score(y, p),
    }

### 4.d — Test de causalidad de Granger

In [ ]:
def test_granger(objetivo: pd.Series, causa: pd.Series,
                 max_lag: int) -> pd.DataFrame:
    """
    Evalúa si 'causa' precede a 'objetivo' (Granger) sobre
    primeras diferencias. Devuelve p-values por rezago.
    """
    from statsmodels.tsa.stattools import grangercausalitytests
    marco = pd.concat(
        [objetivo.rename("objetivo"),
         causa.rename("causa")], axis=1).dropna()
    marco = marco.diff().dropna()  # estacionariedad
    if len(marco) <= max_lag + 1:
        return pd.DataFrame()
    try:
        resultado = grangercausalitytests(
            marco[["objetivo", "causa"]],
            maxlag=max_lag, verbose=False)
    except Exception as error:
        if DEBUG:
            print(f"[DEBUG] Granger falló: {error}")
        return pd.DataFrame()
    filas = []
    for lag, res in resultado.items():
        p_valor = res[0]["ssr_ftest"][1]
        filas.append({
            "lag": lag,
            "p_value": p_valor,
            "significativo": p_valor < 0.05,
        })
    return pd.DataFrame(filas)


def granger_multiple(objetivo: pd.Series,
                     features: pd.DataFrame,
                     columnas: list,
                     max_lag: int) -> pd.DataFrame:
    """Corre Granger para varias métricas on-chain."""
    marcos = []
    for col in columnas:
        if col not in features.columns:
            continue
        res = test_granger(objetivo, features[col], max_lag)
        if not res.empty:
            res.insert(0, "metrica", col)
            marcos.append(res)
    if not marcos:
        return pd.DataFrame()
    return pd.concat(marcos, ignore_index=True)

### 4.e — Backtesting: clasificación de eventos de estrés

In [ ]:
def clasificar_eventos(y_real: pd.Series, y_pred: pd.Series,
                       umbral: float) -> tuple:
    """Binariza real y predicho según un umbral de estrés."""
    real_bin = (y_real >= umbral).astype(int)
    pred_bin = (y_pred >= umbral).astype(int)
    return real_bin, pred_bin


def metricas_clasificacion(real_bin, pred_bin) -> tuple:
    """Precision, recall, F1 y matriz de confusión."""
    metricas = {
        "precision": precision_score(
            real_bin, pred_bin, zero_division=0),
        "recall": recall_score(
            real_bin, pred_bin, zero_division=0),
        "f1": f1_score(
            real_bin, pred_bin, zero_division=0),
    }
    cm = confusion_matrix(real_bin, pred_bin)
    return metricas, cm


def matriz_a_df(cm) -> pd.DataFrame:
    """Convierte la matriz de confusión a DataFrame legible."""
    etiquetas = ["no_evento", "evento"]
    if cm.shape == (2, 2):
        return pd.DataFrame(
            cm, index=etiquetas, columns=etiquetas)
    return pd.DataFrame(cm)

### 4.f — Guardado de reportes

In [ ]:
def guardar_csv(df: pd.DataFrame, nombre: str,
                indice: bool = False) -> None:
    """Guarda un reporte en outputs/reportes/ si procede."""
    if not SAVE_DATA:
        return
    os.makedirs(DIR_REPORTES, exist_ok=True)
    ruta = os.path.join(DIR_REPORTES, nombre)
    df.to_csv(ruta, index=indice)
    if DEBUG:
        print(f"[DEBUG] guardado: {ruta}")

# 5. Proceso

In [ ]:
# 1. Métricas de error sobre el conjunto de test
pred = cargar_predicciones(ARCHIVO_PRED)
test = pred[pred["conjunto"] == "test"] \
    if not pred.empty else pred
error = metricas_error(test)
if DEBUG:
    print("[DEBUG] error test:", error)

In [ ]:
# 2. Backtesting: clasificación de eventos de estrés
resumen = dict(error)
pred_vs_real = pd.DataFrame()
if not test.empty:
    umbral = test["y_real"].quantile(CUANTIL_EVENTO)
    real_bin, pred_bin = clasificar_eventos(
        test["y_real"], test["y_pred"], umbral)
    clasif, cm = metricas_clasificacion(real_bin, pred_bin)
    resumen.update(clasif)
    resumen["umbral_evento"] = umbral
    pred_vs_real = test.copy()
    pred_vs_real["evento_real"] = real_bin.values
    pred_vs_real["evento_pred"] = pred_bin.values
    if DEBUG:
        print("[DEBUG] clasificación:", clasif)
        print(matriz_a_df(cm))
    guardar_csv(matriz_a_df(cm), "val_matriz_confusion.csv",
                indice=True)
    guardar_csv(pred_vs_real, "val_pred_vs_real.csv")

if resumen:
    guardar_csv(pd.DataFrame([resumen]), "val_metricas.csv")

In [ ]:
# 3. Test de causalidad de Granger (on-chain -> FRED)
df = cargar_onchain(ARCHIVO_ONCHAIN)
fred = cargar_fred_ultimo(DIR_EXTERNAL)
features = metricas_onchain(df)
objetivo = construir_objetivo(fred, SERIE_OBJETIVO)

granger = pd.DataFrame()
if not features.empty and not objetivo.empty:
    obj_diario = alinear_diario(objetivo, features.index)
    granger = granger_multiple(
        obj_diario, features, METRICAS_GRANGER,
        MAX_LAG_GRANGER)
    if DEBUG and not granger.empty:
        print(granger)
    guardar_csv(granger, "val_granger.csv")

if DEBUG:
    print("[DEBUG] Validación finalizada")